# External Validation on CARD Dataset

This notebook loads the saved C4 training artifacts (feature schema, scaler, models), prepares the CARD dataset to match the training feature space, applies the trained models, and reports performance and outputs predictions.

- Leakage-free: uses the saved scaler (fitted on C4 train) and does not refit on CARD
- Strict feature alignment: uses the exact feature names and order from training
- Robust preprocessing: replicates questionnaire scoring and key feature engineering used in C4



In [ ]:
# Imports and config
import os
import json
import joblib
import numpy as np
import pandas as pd

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, roc_curve

# Paths
ARTIFACT_DIR = '/Users/eb2007/playground/bullpy/c4_play2/models/cross_validation'
FEATURE_INFO_PATH = os.path.join(ARTIFACT_DIR, 'feature_info_original.json')
SCALER_PATH = os.path.join(ARTIFACT_DIR, 'scaler_original.joblib')
MODELS = {
    'Logistic Regression': os.path.join(ARTIFACT_DIR, 'logistic_regression_original.joblib'),
    'Random Forest': os.path.join(ARTIFACT_DIR, 'random_forest_original.joblib'),
    'XGBoost': os.path.join(ARTIFACT_DIR, 'xgboost_original.joblib'),
    'LightGBM': os.path.join(ARTIFACT_DIR, 'lightgbm_original.joblib'),
    'Gradient Boosting': os.path.join(ARTIFACT_DIR, 'gradient_boosting_original.joblib'),
}

# CARD dataset path
DATA_PATH = '/Users/eb2007/Library/CloudStorage/OneDrive-UniversityofCambridge/Documents/PhD/data/CARD_Noc2025.xlsx'

np.random.seed(42)



In [ ]:
# Load training artifacts (feature schema, scaler, models)
with open(FEATURE_INFO_PATH, 'r') as f:
    feature_info = json.load(f)

feature_names = feature_info['feature_names']
excluded_features = set(feature_info.get('excluded_features', []))
print(f"Loaded feature schema with {len(feature_names)} features")

scaler = joblib.load(SCALER_PATH)
print("Loaded saved StandardScaler (trained on C4)")

loaded_models = {}
for name, path in MODELS.items():
    if os.path.exists(path):
        loaded_models[name] = joblib.load(path)
print(f"Loaded {len(loaded_models)} trained models: {list(loaded_models.keys())}")



In [ ]:
# Load CARD dataset
print(f"Loading CARD data from: {DATA_PATH}")
if DATA_PATH.lower().endswith(('.xlsx', '.xls')):
    df_card = pd.read_excel(DATA_PATH)
else:
    df_card = pd.read_csv(DATA_PATH)
print(df_card.shape)
print("Columns:", list(df_card.columns)[:30], '...')

# Basic cleaning
if 'userid' in df_card.columns:
    df_card = df_card.drop_duplicates(subset=['userid'])
else:
    df_card = df_card.drop_duplicates()

# Ensure expected types are compatible
if 'occupation' in df_card.columns:
    df_card['occupation'] = df_card['occupation'].astype(str)

# Coerce numeric-looking columns
for col in df_card.columns:
    if df_card[col].dtype == 'object':
        try:
            df_card[col] = pd.to_numeric(df_card[col])
        except Exception:
            pass

print("After basic cleaning:", df_card.shape)



In [ ]:
# Questionnaire scoring (replicate C4 rules)
# SPQ-10 items: 1->3, 2->2, 3->1, 4->0; total 0-30
spq_cols = [c for c in df_card.columns if c.lower().startswith('spq_')]
for c in spq_cols:
    df_card[c] = df_card[c].map({1: 3, 2: 2, 3: 1, 4: 0})
if spq_cols:
    df_card['spq_total'] = df_card[spq_cols].sum(axis=1)

# EQ-10 items: 1->1, 2/3/4->0; total 0-10
eq_cols = [c for c in df_card.columns if c.lower().startswith('eq_')]
for c in eq_cols:
    df_card[c] = df_card[c].map({1: 1, 2: 0, 3: 0, 4: 0})
if eq_cols:
    df_card['eq_total'] = df_card[eq_cols].sum(axis=1)

# SQR-10 items: 1->1, 2/3/4->0; total 0-10
sqr_cols = [c for c in df_card.columns if c.lower().startswith('sqr_')]
for c in sqr_cols:
    df_card[c] = df_card[c].map({1: 1, 2: 0, 3: 0, 4: 0})
if sqr_cols:
    df_card['sqr_total'] = df_card[sqr_cols].sum(axis=1)

# AQ-10 items: 1->1, 2/3/4->0; total 0-10
aq_cols = [c for c in df_card.columns if c.lower().startswith('aq_')]
for c in aq_cols:
    df_card[c] = df_card[c].map({1: 1, 2: 0, 3: 0, 4: 0})
if aq_cols:
    df_card['aq_total'] = df_card[aq_cols].sum(axis=1)

print('Scoring complete:')
print({
    'spq_items': len(spq_cols),
    'eq_items': len(eq_cols),
    'sqr_items': len(sqr_cols),
    'aq_items': len(aq_cols)
})



In [ ]:
# Feature engineering to match training
# Age groups and transforms
if 'age' in df_card.columns:
    df_card['sqrt_age'] = np.sqrt(np.clip(df_card['age'], a_min=0, a_max=None))
    df_card['age_group_19-30'] = ((df_card['age'] >= 19) & (df_card['age'] <= 30)).astype(int)
    df_card['age_group_31-45'] = ((df_card['age'] >= 31) & (df_card['age'] <= 45)).astype(int)
    df_card['age_group_46-60'] = ((df_card['age'] >= 46) & (df_card['age'] <= 60)).astype(int)
    df_card['age_group_61+'] = (df_card['age'] >= 61).astype(int)
else:
    df_card['sqrt_age'] = 0.0
    df_card['age_group_19-30'] = 0
    df_card['age_group_31-45'] = 0
    df_card['age_group_46-60'] = 0
    df_card['age_group_61+'] = 0

# sex_num mapping as used in C4 (fallback to 0 if missing)
if 'sex' in df_card.columns:
    df_card['sex_num'] = df_card['sex'].map({1: 0, 2: 1, 3: 2, 4: 3}).fillna(0).astype(int)
else:
    df_card['sex_num'] = 0

# STEM occupation
if 'occupation' in df_card.columns:
    df_card['is_stem_occupation'] = df_card['occupation'].str.contains(
        'science|technology|engineering|math|computer|software|data|research', case=False, na=False
    ).astype(int)
else:
    df_card['is_stem_occupation'] = 0

# Interactions and ratios consistent with feature_info
if {'age', 'eq_total'}.issubset(df_card.columns):
    df_card['age_x_eq'] = df_card['age'] * df_card['eq_total']
else:
    df_card['age_x_eq'] = 0.0

if {'eq_total', 'sqr_total'}.issubset(df_card.columns):
    df_card['eq_sqr_ratio'] = df_card['eq_total'] / (df_card['sqr_total'].replace(0, np.nan))
    df_card['eq_sqr_ratio'] = df_card['eq_sqr_ratio'].replace([np.inf, -np.inf], np.nan).fillna(0.0)
else:
    df_card['eq_sqr_ratio'] = 0.0

# d_score (difference between SQR and EQ; sign consistent with prior code)
if {'sqr_total', 'eq_total'}.issubset(df_card.columns):
    df_card['d_score'] = df_card['sqr_total'] - df_card['eq_total']
else:
    df_card['d_score'] = 0.0



In [ ]:
# Build aligned feature matrix in exact training order
X_card = pd.DataFrame(index=df_card.index)
missing_from_card = []
for fname in feature_names:
    if fname in df_card.columns:
        X_card[fname] = df_card[fname]
    else:
        # Create missing feature as 0
        X_card[fname] = 0
        missing_from_card.append(fname)

# Basic type coercion and missing handling
for c in X_card.columns:
    if X_card[c].dtype == 'object':
        X_card[c] = pd.Categorical(X_card[c]).codes

X_card = X_card.apply(pd.to_numeric, errors='coerce')
num_missing = int(X_card.isnull().sum().sum())
if num_missing > 0:
    X_card = X_card.fillna(X_card.median(numeric_only=True))

print(f"Aligned feature matrix shape: {X_card.shape}")
if missing_from_card:
    print(f"Note: Missing {len(missing_from_card)} features in CARD, filled with 0: {missing_from_card[:10]}...")



In [ ]:
# Apply saved scaler (no refit)
X_card_scaled = scaler.transform(X_card.values)

# Predict with each model
results = {}
probas = {}
for name, model in loaded_models.items():
    y_proba = model.predict_proba(X_card_scaled)[:, 1]
    y_pred = (y_proba >= 0.5).astype(int)
    probas[name] = y_proba
    results[name] = {'n': len(y_pred)}

print('Predictions generated for models:', list(results.keys()))



In [ ]:
# Evaluate if ground-truth label present
metrics_df = None
label_col_candidates = ['autism_target', 'diagnosis_autism', 'has_autism']
label_col = next((c for c in label_col_candidates if c in df_card.columns), None)

if label_col is not None:
    y_true = df_card[label_col].astype(int).clip(0, 1).values
    for name, model in loaded_models.items():
        y_proba = probas[name]
        y_pred = (y_proba >= 0.5).astype(int)
        results[name].update({
            'accuracy': accuracy_score(y_true, y_pred),
            'precision': precision_score(y_true, y_pred, zero_division=0),
            'recall': recall_score(y_true, y_pred, zero_division=0),
            'f1': f1_score(y_true, y_pred, zero_division=0),
            'auc': roc_auc_score(y_true, y_proba)
        })
    metrics_df = pd.DataFrame(results).T
    print('External validation metrics (CARD):')
    print(metrics_df.round(4).sort_values('auc', ascending=False))
else:
    print('No label column found; skipping metrics. Saving predictions only.')



In [ ]:
# Save outputs
os.makedirs('/Users/eb2007/playground/bullpy/c4_play2/data/processed', exist_ok=True)

pred_df = pd.DataFrame({'userid': df_card['userid'] if 'userid' in df_card.columns else np.arange(len(df_card))})
for name, y_proba in probas.items():
    pred_df[f'proba_{name.replace(" ", "_").lower()}'] = y_proba

pred_path = '/Users/eb2007/playground/bullpy/c4_play2/data/processed/card_external_predictions.csv'
pred_df.to_csv(pred_path, index=False)
print(f"Predictions saved to: {pred_path}")

feat_used_path = '/Users/eb2007/playground/bullpy/c4_play2/data/processed/card_features_used.json'
with open(feat_used_path, 'w') as f:
    json.dump({'feature_names': feature_names, 'missing_filled_zero': missing_from_card}, f, indent=2)
print(f"Feature alignment info saved to: {feat_used_path}")

if metrics_df is not None:
    metrics_path = '/Users/eb2007/playground/bullpy/c4_play2/data/processed/card_external_metrics.csv'
    metrics_df.to_csv(metrics_path)
    print(f"Metrics saved to: {metrics_path}")

